In [11]:
!python -m pip install geopy

  Using cached geopy-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.5.0-py3-none-any.whl (114 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)

   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]



In [20]:
import pandas as pd
import numpy as np
import os

In [21]:
locations = pd.read_csv(
    "../data/processed/location_master.csv"
)

print("APY location combinations:", len(locations))
locations.head()

APY location combinations: 771


,State,District,Latitude,Longitude
0,Andaman and Nicobar Island,NICOBARS,NaN,NaN
1,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,NaN,NaN
2,Andaman and Nicobar Island,SOUTH ANDAMANS,NaN,NaN
3,Andhra Pradesh,ADILABAD,NaN,NaN
4,Andhra Pradesh,ANANTAPUR,NaN,NaN


In [22]:
def normalize_name(value):
    if pd.isna(value):
        return None
    
    value = str(value).strip().upper()
    
    # Normalize common punctuation
    value = value.replace("&", "AND")
    value = value.replace("-", " ")
    
    # Remove repeated spaces
    value = " ".join(value.split())
    
    return value

In [23]:
locations["State_Normalized"] = (
    locations["State"].apply(normalize_name)
)

locations["District_Normalized"] = (
    locations["District"].apply(normalize_name)
)

In [24]:
locations["Latitude"] = np.nan
locations["Longitude"] = np.nan
locations["Mapping_Status"] = "UNVERIFIED"
locations["Mapping_Source"] = ""

In [25]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(
    user_agent="smart-crop-ai-college-project"
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1
)

In [26]:
def geocode_verified(district, state):
    
    queries = [
        f"{district}, {state}, India",
        f"{district}, India"
    ]
    
    for query in queries:
        
        try:
            result = geocode(query)
            
            if result is None:
                continue
            
            address = result.address.upper()
            
            expected_state = normalize_name(state)
            
            # State must appear in returned address
            if expected_state in address:
                return {
                    "latitude": result.latitude,
                    "longitude": result.longitude,
                    "status": "VERIFIED",
                    "source": "Nominatim_state_match",
                    "matched_address": result.address
                }
            
        except Exception as e:
            print("Error:", e)
    
    return {
        "latitude": np.nan,
        "longitude": np.nan,
        "status": "REVIEW_REQUIRED",
        "source": "No_verified_match",
        "matched_address": ""
    }

In [27]:
test_cases = [
    ("ADILABAD", "Andhra Pradesh"),
    ("ANANTAPUR", "Andhra Pradesh"),
    ("NICOBARS", "Andaman and Nicobar Island"),
    ("NORTH AND MIDDLE ANDAMAN", "Andaman and Nicobar Island"),
    ("SOUTH ANDAMANS", "Andaman and Nicobar Island")
]

for district, state in test_cases:
    
    result = geocode_verified(
        district,
        state
    )
    
    print(
        state,
        "|",
        district,
        "→",
        result
    )

Andhra Pradesh | ADILABAD → {'latitude': nan, 'longitude': nan, 'status': 'REVIEW_REQUIRED', 'source': 'No_verified_match', 'matched_address': ''}
Andhra Pradesh | ANANTAPUR → {'latitude': 14.6783221, 'longitude': 77.6065039, 'status': 'VERIFIED', 'source': 'Nominatim_state_match', 'matched_address': 'Anantapur, Anantapuram, Andhra Pradesh, 515001, India'}
Andaman and Nicobar Island | NICOBARS → {'latitude': nan, 'longitude': nan, 'status': 'REVIEW_REQUIRED', 'source': 'No_verified_match', 'matched_address': ''}
Andaman and Nicobar Island | NORTH AND MIDDLE ANDAMAN → {'latitude': 12.6112387, 'longitude': 92.8316541, 'status': 'VERIFIED', 'source': 'Nominatim_state_match', 'matched_address': 'North and Middle Andaman, Andaman and Nicobar Islands, India'}
Andaman and Nicobar Island | SOUTH ANDAMANS → {'latitude': nan, 'longitude': nan, 'status': 'REVIEW_REQUIRED', 'source': 'No_verified_match', 'matched_address': ''}


In [28]:
results = []

for index, row in locations.iterrows():

    print(
        f"Processing {index + 1}/{len(locations)}: "
        f"{row['State']} | {row['District']}"
    )

    result = geocode_verified(
        row["District"],
        row["State"]
    )

    results.append({
        "State": row["State"],
        "District": row["District"],
        "Latitude": result["latitude"],
        "Longitude": result["longitude"],
        "Mapping_Status": result["status"],
        "Mapping_Source": result["source"],
        "Matched_Address": result["matched_address"]
    })

Processing 1/771: Andaman and Nicobar Island | NICOBARS
Processing 2/771: Andaman and Nicobar Island | NORTH AND MIDDLE ANDAMAN
Processing 3/771: Andaman and Nicobar Island | SOUTH ANDAMANS
Processing 4/771: Andhra Pradesh | ADILABAD
Processing 5/771: Andhra Pradesh | ANANTAPUR
Processing 6/771: Andhra Pradesh | CHITTOOR
Processing 7/771: Andhra Pradesh | EAST GODAVARI
Processing 8/771: Andhra Pradesh | GUNTUR
Processing 9/771: Andhra Pradesh | HYDERABAD
Processing 10/771: Andhra Pradesh | KADAPA
Processing 11/771: Andhra Pradesh | KARIMNAGAR
Processing 12/771: Andhra Pradesh | KHAMMAM
Processing 13/771: Andhra Pradesh | KRISHNA
Processing 14/771: Andhra Pradesh | KURNOOL
Processing 15/771: Andhra Pradesh | MAHBUBNAGAR
Processing 16/771: Andhra Pradesh | MEDAK
Processing 17/771: Andhra Pradesh | NALGONDA
Processing 18/771: Andhra Pradesh | NIZAMABAD
Processing 19/771: Andhra Pradesh | PRAKASAM
Processing 20/771: Andhra Pradesh | RANGAREDDI
Processing 21/771: Andhra Pradesh | SPSR NELLO

In [33]:
location_results = pd.DataFrame(results)

print("Total locations:", len(location_results))

location_results.head()

Total locations: 771


,State,District,Latitude,Longitude,Mapping_Status,Mapping_Source,Matched_Address
0,Andaman and Nicobar Island,NICOBARS,NaN,NaN,REVIEW_REQUIRED,No_verified_match,
1,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,12.611239,92.831654,VERIFIED,Nominatim_state_match,"North and Middle Andaman, Andaman and Nicobar ..."
2,Andaman and Nicobar Island,SOUTH ANDAMANS,NaN,NaN,REVIEW_REQUIRED,No_verified_match,
3,Andhra Pradesh,ADILABAD,NaN,NaN,REVIEW_REQUIRED,No_verified_match,
4,Andhra Pradesh,ANANTAPUR,14.678322,77.606504,VERIFIED,Nominatim_state_match,"Anantapur, Anantapuram, Andhra Pradesh, 515001..."


In [31]:
print(
    location_results["Mapping_Status"]
    .value_counts()
)

Mapping_Status
VERIFIED           709
REVIEW_REQUIRED     62
Name: count, dtype: int64


In [34]:
verified = (
    location_results["Mapping_Status"] == "VERIFIED"
).sum()

total = len(location_results)

success_rate = verified / total * 100

print(f"Verified: {verified}/{total}")
print(f"Success rate: {success_rate:.2f}%")

Verified: 709/771
Success rate: 91.96%


In [35]:
review_locations = location_results[
    location_results["Mapping_Status"] == "REVIEW_REQUIRED"
].copy()

print("Locations requiring review:", len(review_locations))

review_locations[
    ["State", "District"]
].head(50)

Locations requiring review: 62


,State,District
0,Andaman and Nicobar Island,NICOBARS
2,Andaman and Nicobar Island,SOUTH ANDAMANS
3,Andhra Pradesh,ADILABAD
10,Andhra Pradesh,KARIMNAGAR
15,Andhra Pradesh,MEDAK
16,Andhra Pradesh,NALGONDA
19,Andhra Pradesh,RANGAREDDI
22,Andhra Pradesh,VISAKHAPATANAM
24,Andhra Pradesh,WARANGAL
34,Arunachal Pradesh,LEPARADA


In [36]:
review_by_state = (
    review_locations
    .groupby("State")
    .size()
    .sort_values(ascending=False)
)

print(review_by_state)

State
Bihar                         14
Madhya Pradesh                12
Uttar Pradesh                 10
Andhra Pradesh                 7
Uttarakhand                    3
Andaman and Nicobar Island     2
Punjab                         2
West Bengal                    2
Gujarat                        2
Haryana                        1
Jammu and Kashmir              1
Jharkhand                      1
Delhi                          1
Telangana                      1
Assam                          1
Arunachal Pradesh              1
Himachal Pradesh               1
dtype: int64


In [37]:
os.makedirs("../data/processed", exist_ok=True)

In [38]:
location_results.to_csv(
    "../data/processed/location_reference_results.csv",
    index=False
)

review_locations.to_csv(
    "../data/processed/location_review.csv",
    index=False
)

print("Location reference files saved.")

Location reference files saved.


In [39]:
location_results["Mapping_Status"].value_counts()

Mapping_Status
VERIFIED           709
REVIEW_REQUIRED     62
Name: count, dtype: int64

In [40]:
review_by_state

State
Bihar                         14
Madhya Pradesh                12
Uttar Pradesh                 10
Andhra Pradesh                 7
Uttarakhand                    3
Andaman and Nicobar Island     2
Punjab                         2
West Bengal                    2
Gujarat                        2
Haryana                        1
Jammu and Kashmir              1
Jharkhand                      1
Delhi                          1
Telangana                      1
Assam                          1
Arunachal Pradesh              1
Himachal Pradesh               1
dtype: int64

In [41]:
review_locations = location_results[
    location_results["Mapping_Status"] == "REVIEW_REQUIRED"
].copy()

print("Locations requiring review:", len(review_locations))

review_locations[
    ["State", "District"]
].head(100)

Locations requiring review: 62


,State,District
0,Andaman and Nicobar Island,NICOBARS
2,Andaman and Nicobar Island,SOUTH ANDAMANS
3,Andhra Pradesh,ADILABAD
10,Andhra Pradesh,KARIMNAGAR
15,Andhra Pradesh,MEDAK
...,...,...
745,Uttarakhand,RUDRA PRAYAG
747,Uttarakhand,UDAM SINGH NAGAR
748,Uttarakhand,UTTAR KASHI
766,West Bengal,PARAGANAS NORTH


In [42]:
review_by_state = (
    review_locations
    .groupby("State")
    .size()
    .sort_values(ascending=False)
)

print(review_by_state)

State
Bihar                         14
Madhya Pradesh                12
Uttar Pradesh                 10
Andhra Pradesh                 7
Uttarakhand                    3
Andaman and Nicobar Island     2
Punjab                         2
West Bengal                    2
Gujarat                        2
Haryana                        1
Jammu and Kashmir              1
Jharkhand                      1
Delhi                          1
Telangana                      1
Assam                          1
Arunachal Pradesh              1
Himachal Pradesh               1
dtype: int64


In [43]:
review_locations.to_csv(
    "../data/processed/location_review.csv",
    index=False
)

print("Review file saved.")

Review file saved.


In [44]:
verified_locations = location_results[
    location_results["Mapping_Status"] == "VERIFIED"
].copy()

verified_locations.to_csv(
    "../data/processed/location_verified.csv",
    index=False
)

print("Verified locations:", len(verified_locations))

Verified locations: 709


In [45]:
pd.set_option("display.max_rows", 100)

review_locations[
    ["State", "District", "Mapping_Status", "Mapping_Source"]
]

,State,District,Mapping_Status,Mapping_Source
0,Andaman and Nicobar Island,NICOBARS,REVIEW_REQUIRED,No_verified_match
2,Andaman and Nicobar Island,SOUTH ANDAMANS,REVIEW_REQUIRED,No_verified_match
3,Andhra Pradesh,ADILABAD,REVIEW_REQUIRED,No_verified_match
10,Andhra Pradesh,KARIMNAGAR,REVIEW_REQUIRED,No_verified_match
15,Andhra Pradesh,MEDAK,REVIEW_REQUIRED,No_verified_match
16,Andhra Pradesh,NALGONDA,REVIEW_REQUIRED,No_verified_match
19,Andhra Pradesh,RANGAREDDI,REVIEW_REQUIRED,No_verified_match
22,Andhra Pradesh,VISAKHAPATANAM,REVIEW_REQUIRED,No_verified_match
24,Andhra Pradesh,WARANGAL,REVIEW_REQUIRED,No_verified_match
34,Arunachal Pradesh,LEPARADA,REVIEW_REQUIRED,No_verified_match


In [46]:
print(review_by_state)

State
Bihar                         14
Madhya Pradesh                12
Uttar Pradesh                 10
Andhra Pradesh                 7
Uttarakhand                    3
Andaman and Nicobar Island     2
Punjab                         2
West Bengal                    2
Gujarat                        2
Haryana                        1
Jammu and Kashmir              1
Jharkhand                      1
Delhi                          1
Telangana                      1
Assam                          1
Arunachal Pradesh              1
Himachal Pradesh               1
dtype: int64


In [47]:
pd.set_option("display.max_rows", 100)

review_locations[
    ["State", "District"]
].sort_values(
    ["State", "District"]
)

,State,District
0,Andaman and Nicobar Island,NICOBARS
2,Andaman and Nicobar Island,SOUTH ANDAMANS
3,Andhra Pradesh,ADILABAD
10,Andhra Pradesh,KARIMNAGAR
15,Andhra Pradesh,MEDAK
16,Andhra Pradesh,NALGONDA
19,Andhra Pradesh,RANGAREDDI
22,Andhra Pradesh,VISAKHAPATANAM
24,Andhra Pradesh,WARANGAL
34,Arunachal Pradesh,LEPARADA


In [48]:
review_locations[
    ["State", "District"]
].sort_values(
    ["State", "District"]
).to_csv(
    "../data/processed/location_review_detailed.csv",
    index=False
)

print("Detailed review list saved.")

Detailed review list saved.


In [49]:
review_locations[
    ["State", "District"]
].sort_values(
    ["State", "District"]
)

,State,District
0,Andaman and Nicobar Island,NICOBARS
2,Andaman and Nicobar Island,SOUTH ANDAMANS
3,Andhra Pradesh,ADILABAD
10,Andhra Pradesh,KARIMNAGAR
15,Andhra Pradesh,MEDAK
16,Andhra Pradesh,NALGONDA
19,Andhra Pradesh,RANGAREDDI
22,Andhra Pradesh,VISAKHAPATANAM
24,Andhra Pradesh,WARANGAL
34,Arunachal Pradesh,LEPARADA


In [50]:
print(review_by_state)

State
Bihar                         14
Madhya Pradesh                12
Uttar Pradesh                 10
Andhra Pradesh                 7
Uttarakhand                    3
Andaman and Nicobar Island     2
Punjab                         2
West Bengal                    2
Gujarat                        2
Haryana                        1
Jammu and Kashmir              1
Jharkhand                      1
Delhi                          1
Telangana                      1
Assam                          1
Arunachal Pradesh              1
Himachal Pradesh               1
dtype: int64


In [51]:
historical_state_mapping = {

    # Historical Andhra Pradesh → current Telangana
    ("Andhra Pradesh", "ADILABAD"): ("Telangana", "Adilabad"),
    ("Andhra Pradesh", "KARIMNAGAR"): ("Telangana", "Karimnagar"),
    ("Andhra Pradesh", "MEDAK"): ("Telangana", "Medak"),
    ("Andhra Pradesh", "NALGONDA"): ("Telangana", "Nalgonda"),
    ("Andhra Pradesh", "RANGAREDDI"): ("Telangana", "Rangareddy"),
    ("Andhra Pradesh", "WARANGAL"): ("Telangana", "Warangal"),

    # Historical Andhra Pradesh spelling/location
    ("Andhra Pradesh", "VISAKHAPATANAM"): ("Andhra Pradesh", "Visakhapatnam"),

    # Historical Bihar → current Jharkhand
    ("Bihar", "BOKARO"): ("Jharkhand", "Bokaro"),
    ("Bihar", "DHANBAD"): ("Jharkhand", "Dhanbad"),
    ("Bihar", "DUMKA"): ("Jharkhand", "Dumka"),
    ("Bihar", "EAST SINGHBUM"): ("Jharkhand", "East Singhbhum"),
    ("Bihar", "GARHWA"): ("Jharkhand", "Garhwa"),
    ("Bihar", "GIRIDIH"): ("Jharkhand", "Giridih"),
    ("Bihar", "GODDA"): ("Jharkhand", "Godda"),
    ("Bihar", "GUMLA"): ("Jharkhand", "Gumla"),
    ("Bihar", "HAZARIBAGH"): ("Jharkhand", "Hazaribagh"),
    ("Bihar", "KODERMA"): ("Jharkhand", "Koderma"),
    ("Bihar", "LOHARDAGA"): ("Jharkhand", "Lohardaga"),
    ("Bihar", "PAKUR"): ("Jharkhand", "Pakur"),
    ("Bihar", "PALAMU"): ("Jharkhand", "Palamu"),
    ("Bihar", "WEST SINGHBHUM"): ("Jharkhand", "West Singhbhum"),

    # Historical Madhya Pradesh → current Chhattisgarh
    ("Madhya Pradesh", "BASTAR"): ("Chhattisgarh", "Bastar"),
    ("Madhya Pradesh", "DANTEWADA"): ("Chhattisgarh", "Dantewada"),
    ("Madhya Pradesh", "DHAMTARI"): ("Chhattisgarh", "Dhamtari"),
    ("Madhya Pradesh", "DURG"): ("Chhattisgarh", "Durg"),
    ("Madhya Pradesh", "JANJGIR-CHAMPA"): ("Chhattisgarh", "Janjgir-Champa"),
    ("Madhya Pradesh", "JASHPUR"): ("Chhattisgarh", "Jashpur"),
    ("Madhya Pradesh", "KABIRDHAM"): ("Chhattisgarh", "Kabirdham"),
    ("Madhya Pradesh", "KANKER"): ("Chhattisgarh", "Kanker"),
    ("Madhya Pradesh", "KORBA"): ("Chhattisgarh", "Korba"),
    ("Madhya Pradesh", "MAHASAMUND"): ("Chhattisgarh", "Mahasamund"),
    ("Madhya Pradesh", "RAJNANDGAON"): ("Chhattisgarh", "Rajnandgaon"),
    ("Madhya Pradesh", "SURGUJA"): ("Chhattisgarh", "Surguja"),

    # Historical Uttar Pradesh → current Uttarakhand
    ("Uttar Pradesh", "ALMORA"): ("Uttarakhand", "Almora"),
    ("Uttar Pradesh", "CHAMOLI"): ("Uttarakhand", "Chamoli"),
    ("Uttar Pradesh", "CHAMPAWAT"): ("Uttarakhand", "Champawat"),
    ("Uttar Pradesh", "PAURI GARHWAL"): ("Uttarakhand", "Pauri Garhwal"),
    ("Uttar Pradesh", "PITHORAGARH"): ("Uttarakhand", "Pithoragarh"),
    ("Uttar Pradesh", "RUDRA PRAYAG"): ("Uttarakhand", "Rudraprayag"),
    ("Uttar Pradesh", "TEHRI GARHWAL"): ("Uttarakhand", "Tehri Garhwal"),
    ("Uttar Pradesh", "UDAM SINGH NAGAR"): ("Uttarakhand", "Udham Singh Nagar"),

    # Current-name spelling / administrative changes
    ("Andaman and Nicobar Island", "NICOBARS"): (
        "Andaman and Nicobar Islands",
        "Nicobar"
    ),

    ("Andaman and Nicobar Island", "SOUTH ANDAMANS"): (
        "Andaman and Nicobar Islands",
        "South Andaman"
    ),

    ("Arunachal Pradesh", "LEPARADA"): (
        "Arunachal Pradesh",
        "Lepa Rada"
    ),

    ("Assam", "SOUTH SALMARA MANCACHAR"): (
        "Assam",
        "South Salmara-Mankachar"
    ),

    ("Gujarat", "CHHOTAUDEPUR"): (
        "Gujarat",
        "Chhota Udaipur"
    ),

    ("Haryana", "CHARKI DADRI"): (
        "Haryana",
        "Charkhi Dadri"
    ),

    ("Himachal Pradesh", "LAHUL AND SPITI"): (
        "Himachal Pradesh",
        "Lahaul and Spiti"
    ),

    ("Jammu and Kashmir", "LEH LADAKH"): (
        "Ladakh",
        "Leh"
    ),

    ("Punjab", "FIROZEPUR"): (
        "Punjab",
        "Firozpur"
    ),

    ("Punjab", "SHAHID BHAGAT SINGH NAGAR"): (
        "Punjab",
        "Shaheed Bhagat Singh Nagar"
    ),

    ("West Bengal", "PARAGANAS NORTH"): (
        "West Bengal",
        "North 24 Parganas"
    ),

    ("West Bengal", "PARAGANAS SOUTH"): (
        "West Bengal",
        "South 24 Parganas"
    ),

    # UP districts that remain Uttar Pradesh
    ("Uttar Pradesh", "KUSHI NAGAR"): (
        "Uttar Pradesh",
        "Kushinagar"
    ),

    ("Uttar Pradesh", "SANT KABEER NAGAR"): (
        "Uttar Pradesh",
        "Sant Kabir Nagar"
    ),

    # Current Uttarakhand spellings
    ("Uttarakhand", "RUDRA PRAYAG"): (
        "Uttarakhand",
        "Rudraprayag"
    ),

    ("Uttarakhand", "UDAM SINGH NAGAR"): (
        "Uttarakhand",
        "Udham Singh Nagar"
    ),

    ("Uttarakhand", "UTTAR KASHI"): (
        "Uttarakhand",
        "Uttarkashi"
    ),

    # Telangana record
    ("Telangana", "RANGAREDDI"): (
        "Telangana",
        "Rangareddy"
    ),

    # Jharkhand record
    ("Jharkhand", "EAST SINGHBUM"): (
        "Jharkhand",
        "East Singhbhum"
    ),

    # Delhi
    ("Delhi", "DELHI_TOTAL"): (
        "Delhi",
        "Delhi"
    ),
}

In [52]:
def reconcile_location(state, district):

    key = (state, district)

    if key in historical_state_mapping:

        current_state, current_district = \
            historical_state_mapping[key]

        return pd.Series({
            "Current_State": current_state,
            "Current_District": current_district,
            "Reconciliation_Status": "MAPPED"
        })

    return pd.Series({
        "Current_State": state,
        "Current_District": district,
        "Reconciliation_Status": "UNCHANGED"
    })

In [53]:
reconciled = locations.apply(
    lambda row: reconcile_location(
        row["State"],
        row["District"]
    ),
    axis=1
)

locations = pd.concat(
    [locations, reconciled],
    axis=1
)

In [54]:
review_keys = set(
    zip(
        review_locations["State"],
        review_locations["District"]
    )
)

locations[
    locations.apply(
        lambda row:
        (row["State"], row["District"]) in review_keys,
        axis=1
    )
][
    [
        "State",
        "District",
        "Current_State",
        "Current_District",
        "Reconciliation_Status"
    ]
]

,State,District,Current_State,Current_District,Reconciliation_Status
0,Andaman and Nicobar Island,NICOBARS,Andaman and Nicobar Islands,Nicobar,MAPPED
2,Andaman and Nicobar Island,SOUTH ANDAMANS,Andaman and Nicobar Islands,South Andaman,MAPPED
3,Andhra Pradesh,ADILABAD,Telangana,Adilabad,MAPPED
10,Andhra Pradesh,KARIMNAGAR,Telangana,Karimnagar,MAPPED
15,Andhra Pradesh,MEDAK,Telangana,Medak,MAPPED
16,Andhra Pradesh,NALGONDA,Telangana,Nalgonda,MAPPED
19,Andhra Pradesh,RANGAREDDI,Telangana,Rangareddy,MAPPED
22,Andhra Pradesh,VISAKHAPATANAM,Andhra Pradesh,Visakhapatnam,MAPPED
24,Andhra Pradesh,WARANGAL,Telangana,Warangal,MAPPED
34,Arunachal Pradesh,LEPARADA,Arunachal Pradesh,Lepa Rada,MAPPED


In [55]:
def geocode_reconciled(state, district):

    queries = [
        f"{district}, {state}, India",
        f"{district} district, {state}, India"
    ]

    for query in queries:

        try:
            result = geocode(query)

            if result:
                address = result.address.upper()

                return {
                    "Latitude": result.latitude,
                    "Longitude": result.longitude,
                    "Geocoded_Address": result.address,
                    "Geocode_Status": "FOUND"
                }

        except Exception as e:
            print("Error:", e)

    return {
        "Latitude": np.nan,
        "Longitude": np.nan,
        "Geocoded_Address": "",
        "Geocode_Status": "NOT_FOUND"
    }

In [56]:
review_mask = locations.apply(
    lambda row:
    (row["State"], row["District"]) in review_keys,
    axis=1
)

review_df = locations[review_mask].copy()

print("Locations to process:", len(review_df))

Locations to process: 62


In [57]:
geo_results = []

for _, row in review_df.iterrows():

    print(
        row["State"],
        "|",
        row["District"],
        "→",
        row["Current_State"],
        "|",
        row["Current_District"]
    )

    result = geocode_reconciled(
        row["Current_State"],
        row["Current_District"]
    )

    geo_results.append(result)

Andaman and Nicobar Island | NICOBARS → Andaman and Nicobar Islands | Nicobar
Andaman and Nicobar Island | SOUTH ANDAMANS → Andaman and Nicobar Islands | South Andaman
Andhra Pradesh | ADILABAD → Telangana | Adilabad
Andhra Pradesh | KARIMNAGAR → Telangana | Karimnagar
Andhra Pradesh | MEDAK → Telangana | Medak
Andhra Pradesh | NALGONDA → Telangana | Nalgonda
Andhra Pradesh | RANGAREDDI → Telangana | Rangareddy
Andhra Pradesh | VISAKHAPATANAM → Andhra Pradesh | Visakhapatnam
Andhra Pradesh | WARANGAL → Telangana | Warangal
Arunachal Pradesh | LEPARADA → Arunachal Pradesh | Lepa Rada
Assam | SOUTH SALMARA MANCACHAR → Assam | South Salmara-Mankachar
Bihar | BOKARO → Jharkhand | Bokaro
Bihar | DHANBAD → Jharkhand | Dhanbad
Bihar | DUMKA → Jharkhand | Dumka
Bihar | EAST SINGHBUM → Jharkhand | East Singhbhum
Bihar | GARHWA → Jharkhand | Garhwa
Bihar | GIRIDIH → Jharkhand | Giridih
Bihar | GODDA → Jharkhand | Godda
Bihar | GUMLA → Jharkhand | Gumla
Bihar | HAZARIBAGH → Jharkhand | Hazaribagh

In [58]:
geo_results_df = pd.DataFrame(geo_results)

review_df = review_df.reset_index(drop=True)

review_df = pd.concat(
    [review_df, geo_results_df],
    axis=1
)

In [59]:
print(
    review_df["Geocode_Status"].value_counts()
)

Geocode_Status
FOUND        61
NOT_FOUND     1
Name: count, dtype: int64


In [60]:
review_df[
    [
        "State",
        "District",
        "Current_State",
        "Current_District",
        "Latitude",
        "Longitude",
        "Geocode_Status"
    ]
]

,State,District,Current_State,Current_District,Latitude,Latitude,Longitude,Longitude,Geocode_Status
0,Andaman and Nicobar Island,NICOBARS,Andaman and Nicobar Islands,Nicobar,NaN,7.000017,NaN,93.811053,FOUND
1,Andaman and Nicobar Island,SOUTH ANDAMANS,Andaman and Nicobar Islands,South Andaman,NaN,10.705690,NaN,92.487468,FOUND
2,Andhra Pradesh,ADILABAD,Telangana,Adilabad,NaN,19.675945,NaN,78.533990,FOUND
3,Andhra Pradesh,KARIMNAGAR,Telangana,Karimnagar,NaN,18.434812,NaN,79.132804,FOUND
4,Andhra Pradesh,MEDAK,Telangana,Medak,NaN,17.937510,NaN,78.211745,FOUND
5,Andhra Pradesh,NALGONDA,Telangana,Nalgonda,NaN,17.050441,NaN,79.266924,FOUND
6,Andhra Pradesh,RANGAREDDI,Telangana,Rangareddy,NaN,16.934415,NaN,77.778834,FOUND
7,Andhra Pradesh,VISAKHAPATANAM,Andhra Pradesh,Visakhapatnam,NaN,17.693553,NaN,83.292130,FOUND
8,Andhra Pradesh,WARANGAL,Telangana,Warangal,NaN,17.982064,NaN,79.597095,FOUND
9,Arunachal Pradesh,LEPARADA,Arunachal Pradesh,Lepa Rada,NaN,27.921127,NaN,94.740936,FOUND


In [63]:
print(review_df["Geocode_Status"].value_counts())

Geocode_Status
FOUND        61
NOT_FOUND     1
Name: count, dtype: int64


In [64]:
review_df[
    review_df["Geocode_Status"] == "NOT_FOUND"
][
    ["State", "District", "Current_State", "Current_District"]
]

,State,District,Current_State,Current_District
27,Gujarat,DOHAD,Gujarat,DOHAD


In [66]:
dohad_result = geocode_reconciled(
    "Gujarat",
    "Dahod"
)

print(dohad_result)

{'Latitude': 22.9194099, 'Longitude': 74.1342725, 'Geocoded_Address': 'Dahod, Gujarat, India', 'Geocode_Status': 'FOUND'}


In [67]:
review_df.loc[
    review_df["District"] == "DOHAD",
    "Current_District"
] = "Dahod"

review_df.loc[
    review_df["District"] == "DOHAD",
    "Latitude"
] = dohad_result["Latitude"]

review_df.loc[
    review_df["District"] == "DOHAD",
    "Longitude"
] = dohad_result["Longitude"]

review_df.loc[
    review_df["District"] == "DOHAD",
    "Geocoded_Address"
] = dohad_result["Geocoded_Address"]

review_df.loc[
    review_df["District"] == "DOHAD",
    "Geocode_Status"
] = dohad_result["Geocode_Status"]

In [68]:
print(review_df["Geocode_Status"].value_counts())

Geocode_Status
FOUND    62
Name: count, dtype: int64


In [69]:
print(
    review_df[
        review_df["Geocode_Status"] != "FOUND"
    ]
)

Empty DataFrame
Columns: [State, District, Latitude, Longitude, State_Normalized, District_Normalized, Mapping_Status, Mapping_Source, Current_State, Current_District, Reconciliation_Status, Latitude, Longitude, Geocoded_Address, Geocode_Status]
Index: []


In [71]:
final_locations = pd.concat(
    [
        verified_locations,
        review_df
    ],
    ignore_index=True
)

print("Final location count:", len(final_locations))
print("Unique State-District pairs:",
      final_locations[["State", "District"]].drop_duplicates().shape[0])

print(
    final_locations["Geocode_Status"]
    .value_counts()
)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [72]:
final_locations.to_csv(
    "../data/processed/location_reference_final.csv",
    index=False
)

print("Final location reference saved!")

NameError: name 'final_locations' is not defined

In [73]:
print("verified_locations duplicate columns:")
print(verified_locations.columns[verified_locations.columns.duplicated()].tolist())

print("\nreview_df duplicate columns:")
print(review_df.columns[review_df.columns.duplicated()].tolist())

verified_locations duplicate columns:
[]

review_df duplicate columns:
['Latitude', 'Longitude']


In [74]:
verified_locations = verified_locations.loc[
    :, ~verified_locations.columns.duplicated()
].copy()

review_df = review_df.loc[
    :, ~review_df.columns.duplicated()
].copy()

In [75]:
print("Verified duplicate columns:",
      verified_locations.columns[
          verified_locations.columns.duplicated()
      ].tolist())

print("Review duplicate columns:",
      review_df.columns[
          review_df.columns.duplicated()
      ].tolist())

Verified duplicate columns: []
Review duplicate columns: []


In [76]:
common_columns = [
    "State",
    "District",
    "Current_State",
    "Current_District",
    "Latitude",
    "Longitude",
    "Geocoded_Address",
    "Geocode_Status"
]

In [77]:
verified_locations_final = verified_locations[
    [col for col in common_columns
     if col in verified_locations.columns]
].copy()

review_locations_final = review_df[
    [col for col in common_columns
     if col in review_df.columns]
].copy()

In [78]:
print(verified_locations_final.columns.tolist())
print(review_locations_final.columns.tolist())

['State', 'District', 'Latitude', 'Longitude']
['State', 'District', 'Current_State', 'Current_District', 'Latitude', 'Longitude', 'Geocoded_Address', 'Geocode_Status']


In [79]:
final_locations = pd.concat(
    [
        verified_locations_final,
        review_locations_final
    ],
    ignore_index=True
)

In [80]:
print("Final location count:", len(final_locations))

print(
    "Unique State-District pairs:",
    final_locations[
        ["State", "District"]
    ].drop_duplicates().shape[0]
)

Final location count: 771
Unique State-District pairs: 771


In [81]:
print("Missing Latitude:",
      final_locations["Latitude"].isna().sum())

print("Missing Longitude:",
      final_locations["Longitude"].isna().sum())

Missing Latitude: 61
Missing Longitude: 61


In [82]:
print(final_locations["Geocode_Status"].value_counts())

Geocode_Status
FOUND    62
Name: count, dtype: int64


In [83]:
duplicates = final_locations[
    final_locations.duplicated(
        subset=["State", "District"],
        keep=False
    )
]

print("Duplicate State-District pairs:",
      len(duplicates))

duplicates.head(20)

Duplicate State-District pairs: 0


,State,District,Latitude,Longitude,Current_State,Current_District,Geocoded_Address,Geocode_Status


In [84]:
final_locations.to_csv(
    "../data/processed/location_reference_final.csv",
    index=False
)

print("✅ Final location reference saved!")
print("Rows:", len(final_locations))

✅ Final location reference saved!
Rows: 771
